# AT-DJ 02b — COT LLM with 02a Feature Map → Best Valid Tanda

This Colab notebook uses the feature artifacts generated by **02a** as the source of truth:

- `df_merged.pkl`
- `feature_catalog.pkl`
- `feature_prompt.txt`

It does **not** create collapsed/engineered features.

## Primary output

The main goal is **not** to return the highest-scoring individual songs. The main goal is to return the **best valid tanda**.

A tanda is valid only if all tracks share the same `combo_key`:

- `style == "tango"` → 4 songs
- `style == "vals"` or `style == "milonga"` → 3 songs

Therefore, the notebook scores individual songs first, then aggregates those scores at the `combo_key` level. For each `combo_key`, it takes the best valid-size subset and computes the tanda score as the **average COT score** of those songs. The highest average-scoring valid set becomes the best tanda.

## Sanity-check output

The notebook also prints:

- top 5–10 individually scored songs, regardless of combo
- bottom 5 individually scored songs, regardless of combo

These are only sanity checks to see whether the LLM-selected feature ranges make musical sense. They are **not** the final recommendation.

## Feature usage rule

COT scoring uses only numeric original features from the 02a feature map, after excluding identity/context fields:

- `title`
- `orchestra`
- `singer`
- `album`
- `filename`
- `combo_key`
- `style`, `year`, `decade` as scoring features

`style`, `year`, and `decade` may still be used as hard constraints when the prompt explicitly asks for them.
`combo_key` is used only for deterministic tanda grouping.


## 0. Imports

In [3]:
from pathlib import Path
import json
import pickle
import re
import time
import os
import warnings
from getpass import getpass
from collections import defaultdict

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)


## 1. Mount Drive and load 02a artifacts

In [4]:
from google.colab import drive
drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# Adjust this if your 02a artifacts are saved elsewhere.
ARTIFACT_DIR = Path("drive/MyDrive/GenAI/proj/atdj_features_exp")

df_path = ARTIFACT_DIR / "df_merged.pkl"
catalog_path = ARTIFACT_DIR / "feature_catalog.pkl"
prompt_path = ARTIFACT_DIR / "feature_prompt.txt"

assert df_path.exists(), f"Missing {df_path}. Please run 02a first."
assert catalog_path.exists(), f"Missing {catalog_path}. Please run 02a first."
assert prompt_path.exists(), f"Missing {prompt_path}. Please run 02a first."

df_merged = pd.read_pickle(df_path)

with open(catalog_path, "rb") as f:
    FEATURE_CATALOG_SELECTED = pickle.load(f)

FEATURE_PROMPT_FROM_02A = prompt_path.read_text(encoding="utf-8")

print("Loaded 02a artifacts")
print("df_merged:", df_merged.shape)
print("feature groups:", list(FEATURE_CATALOG_SELECTED.keys()))
print("feature prompt chars:", len(FEATURE_PROMPT_FROM_02A))


Loaded 02a artifacts
df_merged: (294, 94)
feature groups: ['categorical_metadata', 'numeric_metadata', 'rhythm_tempo', 'harmony_key', 'energy_dynamics', 'spectral_timbre', 'mirex_mood', 'jamendo_genre', 'jamendo_instrument', 'jamendo_mood_function', 'essentia_mood']
feature prompt chars: 9630


## 2. Build the COT scoring feature list from the 02a feature map

In [6]:
# These fields can appear in display output or deterministic constraints,
# but the LLM should not use them as audio-feature scoring dimensions.
IDENTITY_CONTEXT_FIELDS = {
    "title",
    "orchestra",
    "singer",
    "album",
    "filename",
}

# combo_key is not an audio feature; it is the deterministic tanda grouping key.
GROUPING_FIELD = "combo_key"

# These can be parsed as hard filters if the prompt asks for them.
# They should not be range-scored as acoustic features.
HARD_CONSTRAINT_FIELDS = {
    "style",
    "year",
    "decade",
}

EXCLUDE_FROM_COT_SCORING = IDENTITY_CONTEXT_FIELDS | {GROUPING_FIELD} | HARD_CONSTRAINT_FIELDS

def flatten_catalog(feature_catalog: dict) -> dict:
    """Return {feature_name: description} from the 02a selected feature map."""
    out = {}
    for group, feats in feature_catalog.items():
        for fname, desc in feats:
            out[fname] = desc
    return out

FEATURE_DESC_MAP_02A = flatten_catalog(FEATURE_CATALOG_SELECTED)

# Only use fields that:
# 1. came from 02a feature_catalog.pkl
# 2. exist in df_merged
# 3. are numeric
# 4. are not identity/grouping/constraint fields
COT_NUMERIC_FEATURES = [
    f for f in FEATURE_DESC_MAP_02A
    if f in df_merged.columns
    and pd.api.types.is_numeric_dtype(df_merged[f])
    and f not in EXCLUDE_FROM_COT_SCORING
]

print(f"Features in 02a selected catalog: {len(FEATURE_DESC_MAP_02A)}")
print(f"Numeric original features allowed for COT scoring: {len(COT_NUMERIC_FEATURES)}")
print("\nFirst 30 COT scoring features:")
print(COT_NUMERIC_FEATURES[:30])

# Optional sanity check: no identity/context fields should be in COT scoring.
assert not (set(COT_NUMERIC_FEATURES) & EXCLUDE_FROM_COT_SCORING)


Features in 02a selected catalog: 64
Numeric original features allowed for COT scoring: 53

First 30 COT scoring features:
['duration_seconds', 'duration', 'bpm', 'onset_rate', 'danceability', 'is_danceable', 'key_strength', 'chords_changes_rate', 'chords_number_rate', 'hpcp_entropy', 'tuning_frequency', 'average_loudness', 'dynamic_complexity', 'spectral_energy_mean', 'spectral_centroid_mean', 'spectral_centroid_stdev', 'spectral_flux_mean', 'dissonance_mean', 'mirex_mood_rollicking_cheerful_fun_sweet_amiable_good_natured', 'mirex_mood_literate_poignant_wistful_bittersweet_autumnal_brooding', 'mirex_mood_humorous_silly_campy_quirky_whimsical_witty_wry', 'jamendo_alternative', 'jamendo_ambient', 'jamendo_classical', 'jamendo_easylistening', 'jamendo_electronic', 'jamendo_experimental', 'jamendo_folk', 'jamendo_house', 'jamendo_instrumentalpop']


In [7]:
def build_filtered_feature_prompt(df: pd.DataFrame, feature_catalog: dict, allowed_features: list[str]) -> str:
    """Build an LLM prompt section using only allowed 02a-selected scoring features."""
    allowed = set(allowed_features)
    lines = []
    for group, feats in feature_catalog.items():
        group_lines = []
        for fname, desc in feats:
            if fname not in allowed or fname not in df.columns:
                continue
            col = pd.to_numeric(df[fname], errors="coerce")
            if col.notna().sum() == 0:
                continue
            stats = (
                f"range [{col.min():.6g}, {col.max():.6g}], "
                f"mean={col.mean():.6g}, std={col.std():.6g}"
            )
            group_lines.append(f"- **{fname}**: {desc} | {stats}")
        if group_lines:
            lines.append(f"\n### {group.upper()} FEATURES")
            lines.extend(group_lines)
    return "\n".join(lines)

COT_FEATURE_PROMPT = build_filtered_feature_prompt(
    df_merged,
    FEATURE_CATALOG_SELECTED,
    COT_NUMERIC_FEATURES,
)

print(COT_FEATURE_PROMPT[:2500])
print("\nPrompt chars:", len(COT_FEATURE_PROMPT))



### NUMERIC_METADATA FEATURES
- **duration_seconds**: Track duration in seconds from metadata/catalog | range [124, 218.07], mean=166.617, std=18.8916
- **duration**: Track duration in seconds from Essentia | range [124.108, 218.567], mean=166.895, std=18.9297

### RHYTHM_TEMPO FEATURES
- **bpm**: Beats per minute — estimated tempo | range [90.8828, 184.571], mean=120.707, std=12.9288
- **onset_rate**: Number of detected note/onset events per second; higher means denser rhythm | range [1.95661, 5.60483], mean=3.38883, std=0.532919
- **danceability**: Essentia danceability score | range [0.834848, 1.30977], mean=1.02227, std=0.0788446
- **is_danceable**: Essentia classifier probability/score for danceability | range [0.00335481, 0.335515], mean=0.0565611, std=0.0550531

### HARMONY_KEY FEATURES
- **key_strength**: Confidence/strength of the estimated musical key | range [0.525213, 0.909049], mean=0.753929, std=0.0718523
- **chords_changes_rate**: Rate of chord changes; rough harmonic m

## 3. Ensure `combo_key` exists for tanda grouping

In [8]:
def ensure_combo_key(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "combo_key" not in df.columns:
        singer_clean = df["singer"].fillna("Instrumental") if "singer" in df.columns else "Instrumental"
        df["combo_key"] = (
            df["orchestra"].fillna("Unknown").astype(str).str.lower().str.strip()
            + " | "
            + pd.Series(singer_clean).astype(str).str.lower().str.strip()
            + " | "
            + df["style"].fillna("unknown").astype(str).str.lower().str.strip()
        )
        print("Created fallback combo_key from orchestra + singer + style.")
    else:
        print("Using existing combo_key from dataset.")
    return df

df_merged = ensure_combo_key(df_merged)

needed_display_cols = ["title", "orchestra", "singer", "style", "year", "decade", "album", "filename", "combo_key"]
print([c for c in needed_display_cols if c in df_merged.columns])
df_merged[["combo_key", "style"]].head()


Using existing combo_key from dataset.
['title', 'orchestra', 'singer', 'style', 'year', 'decade', 'album', 'filename', 'combo_key']


,combo_key,style
0,angel d'agostino | angel vargas | milonga,milonga
1,angel d'agostino | angel vargas | milonga,milonga
2,angel d'agostino | angel vargas | milonga,milonga
3,angel d'agostino | angel vargas | milonga,milonga
4,angel d'agostino | angel vargas | milonga,milonga


## 4. API key setup

In [9]:
from openai import OpenAI

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")

MODEL = "gpt-4o-mini"
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
print("OpenAI client ready; model =", MODEL)


OpenAI API key: ··········
OpenAI client ready; model = gpt-4o-mini


## 5. Prompt constraint extraction

In [10]:
STYLE_VALUES = {"tango", "vals", "milonga"}

def extract_hard_constraints(prompt: str) -> dict:
    """Extract deterministic constraints from the prompt.

    These constraints are not audio features. They are used to filter the candidate pool
    before scoring / tanda grouping.
    """
    p = prompt.lower()
    constraints = {}

    # style as an optional hard constraint
    for style in STYLE_VALUES:
        if re.search(rf"\b{style}\b", p):
            constraints["style"] = style
            break

    # decade: supports "1940s", "1940", "40s", "50s"
    m = re.search(r"\b(19[2-9]0|20[0-3]0)s?\b", p)
    if m:
        constraints["decade"] = int(m.group(1))
    else:
        m2 = re.search(r"\b([2-9]0)s\b", p)
        if m2:
            constraints["decade"] = 1900 + int(m2.group(1))

    # exact year if present
    y = re.search(r"\b(19[2-9][0-9]|20[0-3][0-9])\b", p)
    if y:
        constraints["year"] = int(y.group(1))

    return constraints

for ex in [
    "romantic vals from the 1940s, smooth but still danceable",
    "energetic tango with strong rhythm",
    "bright cheerful milonga from the 50s",
    "warm nostalgic songs from 1942",
]:
    print(ex, "->", extract_hard_constraints(ex))


romantic vals from the 1940s, smooth but still danceable -> {'style': 'vals', 'decade': 1940}
energetic tango with strong rhythm -> {'style': 'tango'}
bright cheerful milonga from the 50s -> {'style': 'milonga', 'decade': 1950}
warm nostalgic songs from 1942 -> {'year': 1942}


## 6. COT feature/range prediction using only 02a-selected original features

In [11]:
COT_SYSTEM_PROMPT = """You are an expert Argentine Tango DJ and audio engineer.
You help match text descriptions of desired music to songs using audio features.

When given a text prompt and a list of available audio features with their observed ranges,
you must select the most relevant features and predict target value ranges that would
match the described music."""

COT_USER_TEMPLATE = """Given a user's text prompt describing desired tango music, select the most
relevant audio features and specify target ranges for song matching.

## Available Features:
{feature_descriptions}

## User Prompt: "{prompt}"

## Instructions
Think step by step:
1. What musical qualities does this prompt describe?
2. Which numeric features from the list above best capture those qualities? Select 5-10 features.
3. For each selected feature, what value range would match the prompt? Use the min/max values within the observed ranges as guidance.
4. Give higher weights to features that directly express the prompt.
5. Validate your reasoning: does this combination of features and ranges make musical sense together?
   If not, refine your selections.

## Important:
- Use only the feature names provided by the user.
- Do not invent features.
- Do not use title, orchestra, singer, album, filename, style, year, decade, or combo_key as audio scoring features.
- Style/year/decade, when relevant, are handled outside your feature scoring as hard constraints.


## Response Format:
Respond ONLY with valid JSON (no markdown, no code blocks):
{{
  "reasoning": "brief explanation",
  "selected_features": {{
    "feature_name": {{
      "min": <float>,
      "max": <float>,
      "direction": "higher_better" | "lower_better" | "range",
      "weight": <float between 0 and 1, importance of this feature>
    }}
  }},
  "validation": "brief self-check"
}}
"""

def extract_json(text: str) -> dict:
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", text, flags=re.S)
        if not m:
            raise
        return json.loads(m.group(0))


def cot_select_and_range(prompt: str) -> dict:
    user_msg = COT_USER_TEMPLATE.format(
        feature_descriptions=COT_FEATURE_PROMPT,
        prompt=prompt,
    )
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": COT_SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ],
        temperature=0.2,
        response_format={"type": "json_object"},
    )
    parsed = extract_json(resp.choices[0].message.content)

    # Keep only valid numeric original features from the 02a-selected feature map.
    # This is NOT a new feature-selection step; it only removes hallucinated/non-allowed fields.
    selected = parsed.get("selected_features", {}) or {}
    selected = {
        f: spec for f, spec in selected.items()
        if f in COT_NUMERIC_FEATURES
    }
    parsed["selected_features"] = selected
    return parsed


## 7. Score songs, then rank valid tandas by combo-level average score

Important distinction:

- Individual song scores are intermediate values.
- Final recommendation is the best valid tanda.
- A valid tanda is formed within one `combo_key` only.
- Tanda score = average score of the selected songs inside that `combo_key`.


In [12]:
def apply_constraints(df: pd.DataFrame, constraints: dict) -> pd.DataFrame:
    """Apply deterministic prompt constraints before scoring.

    These are not COT scoring features. They only restrict the candidate pool.
    """
    out = df.copy()
    if "style" in constraints and "style" in out.columns:
        out = out[out["style"].astype(str).str.lower() == constraints["style"]]
    if "decade" in constraints and "decade" in out.columns:
        out = out[pd.to_numeric(out["decade"], errors="coerce") == constraints["decade"]]
    if "year" in constraints and "year" in out.columns:
        out = out[pd.to_numeric(out["year"], errors="coerce") == constraints["year"]]
    return out


def infer_tanda_size(style_value) -> int:
    """Tango tanda convention used in this experiment."""
    if pd.isna(style_value):
        return 4
    style = str(style_value).lower().strip()
    if style == "tango":
        return 4
    if style in {"vals", "milonga"}:
        return 3
    return 4


def score_feature_value(value, spec, data_min, data_max):
    try:
        v = float(value)
    except Exception:
        return np.nan
    if np.isnan(v):
        return np.nan

    lo = spec.get("min", None)
    hi = spec.get("max", None)
    direction = spec.get("direction", "range")

    try:
        lo = float(lo) if lo is not None else None
        hi = float(hi) if hi is not None else None
    except Exception:
        lo, hi = None, None

    denom = data_max - data_min if data_max > data_min else 1.0
    z = (v - data_min) / denom

    # If the LLM gives no usable range, fall back to directional scoring.
    if lo is None or hi is None or hi <= lo:
        if direction == "lower_better":
            return float(1 - z)
        return float(z)

    lo = max(lo, data_min)
    hi = min(hi, data_max)

    if hi <= lo:
        return 0.0
    if lo <= v <= hi:
        return 1.0

    # Outside the target range, decay linearly by distance to nearest boundary.
    dist = min(abs(v - lo), abs(v - hi))
    return float(max(0, 1 - dist / denom))


def score_dataframe(df: pd.DataFrame, specs: dict) -> pd.DataFrame:
    """Score individual songs using the LLM-selected original numeric features."""
    out = df.copy()
    scores = []
    matched = []

    feature_bounds = {}
    for f in specs:
        col = pd.to_numeric(df_merged[f], errors="coerce")
        feature_bounds[f] = (float(col.min()), float(col.max()))

    for _, row in out.iterrows():
        vals = {}
        total_w = 0.0
        score_sum = 0.0

        for f, spec in specs.items():
            data_min, data_max = feature_bounds[f]
            s = score_feature_value(row.get(f), spec, data_min, data_max)
            if np.isnan(s):
                continue
            w = float(spec.get("weight", 0.5) or 0.5)
            vals[f] = {
                "value": float(row[f]) if pd.notna(row[f]) else None,
                "feature_score": float(s),
                "weight": float(w),
            }
            score_sum += w * s
            total_w += w

        final_score = score_sum / total_w if total_w > 0 else 0.0
        scores.append(final_score)
        matched.append(vals)

    out["_cot_score"] = scores
    out["_matched_features"] = matched
    return out.sort_values("_cot_score", ascending=False)


def row_to_track_dict(r: pd.Series) -> dict:
    cols = [
        "title", "orchestra", "singer", "style", "year", "decade",
        "album", "filename", "combo_key"
    ]
    d = {}
    for c in cols:
        if c in r.index:
            val = r[c]
            if pd.isna(val):
                val = None
            elif isinstance(val, np.generic):
                val = val.item()
            d[c] = val
    d["score"] = float(r["_cot_score"])
    d["matched_features"] = r["_matched_features"]
    return d


def build_ranked_tandas(scored_df: pd.DataFrame, constraints: dict, max_tandas: int | None = None) -> list[dict]:
    """Rank valid tandas by combo-level average score.

    This is the core recommendation step.

    For each combo_key:
    1. infer tanda size from style
    2. require enough songs in that combo
    3. select the top-scoring N songs within that combo
    4. compute tanda_score = average of those N song scores
    5. sort all candidate tandas by tanda_score descending
    """
    if scored_df.empty or "combo_key" not in scored_df.columns:
        return []

    candidates = []

    for combo_key, group in scored_df.groupby("combo_key", dropna=True):
        if group.empty:
            continue

        styles = group["style"].dropna().astype(str).str.lower().unique().tolist() if "style" in group.columns else []
        if len(styles) == 0:
            style = constraints.get("style", "tango")
        elif len(styles) == 1:
            style = styles[0]
        else:
            # combo_key should combine orchestra, singer, and style.
            # If a combo somehow contains multiple styles, skip to avoid invalid tandas.
            continue

        tanda_size = infer_tanda_size(style)
        if len(group) < tanda_size:
            continue

        chosen = group.sort_values("_cot_score", ascending=False).head(tanda_size)
        tanda_score = float(chosen["_cot_score"].mean())

        candidates.append({
            "combo_key": combo_key,
            "style": style,
            "tanda_size": tanda_size,
            "tanda_score": tanda_score,
            "tracks": [row_to_track_dict(r) for _, r in chosen.iterrows()],
            "n_available_in_combo": int(len(group)),
            "score_rule": "mean of selected tracks' individual COT scores",
        })

    candidates = sorted(
        candidates,
        key=lambda x: (x["tanda_score"], x["n_available_in_combo"]),
        reverse=True,
    )
    if max_tandas is not None:
        candidates = candidates[:max_tandas]
    return candidates


def rank_prompt_to_best_tanda_cot(
    prompt: str,
    df: pd.DataFrame,
    top_k: int = 10,
    bottom_k: int = 5,
    top_tanda_k: int = 5,
) -> dict:
    """Run COT feature selection/range prediction, then return the best valid tanda.

    The top/bottom individual songs are included only for sanity checking.
    """
    if top_k < 5 or top_k > 10:
        raise ValueError("top_k should be between 5 and 10 for this experiment.")

    start = time.time()
    constraints = extract_hard_constraints(prompt)

    cot = cot_select_and_range(prompt)
    specs_raw = cot.get("selected_features", {}) or {}
    specs = {f: s for f, s in specs_raw.items() if f in COT_NUMERIC_FEATURES}

    if not specs:
        raise ValueError("LLM selected no valid numeric original features from the 02a feature map.")

    candidate_pool = apply_constraints(df, constraints)
    if candidate_pool.empty:
        print("Warning: hard constraints produced an empty pool; falling back to full dataset.")
        candidate_pool = df.copy()

    scored = score_dataframe(candidate_pool, specs)
    ranked_tandas = build_ranked_tandas(scored, constraints, max_tandas=top_tanda_k)
    best_tanda = ranked_tandas[0] if ranked_tandas else None

    return {
        "prompt": prompt,
        "method": "cot_llm_02a_feature_map_combo_average_best_tanda",
        "primary_output": "best_tanda",
        "hard_constraints": constraints,
        "best_tanda": best_tanda,
        "top_candidate_tandas": ranked_tandas,
        "sanity_check_top_k_songs": [row_to_track_dict(r) for _, r in scored.head(top_k).iterrows()],
        "sanity_check_bottom_k_songs": [row_to_track_dict(r) for _, r in scored.tail(bottom_k).iterrows()],
        "feature_ranges_used": specs,
        "llm_reasoning": cot.get("reasoning", ""),
        "llm_validation": cot.get("validation", ""),
        "metadata": {
            "model": MODEL,
            "latency_seconds": round(time.time() - start, 3),
            "n_candidate_tracks_after_constraints": int(len(candidate_pool)),
            "n_valid_tanda_candidates": int(len(ranked_tandas)),
            "n_numeric_features_scored": int(len(specs)),
            "n_features_selected_raw": int(len(specs_raw)),
            "used_02a_feature_catalog": True,
            "created_collapsed_features": False,
            "tanda_ranking_rule": "rank each combo_key by average score of its best valid-size subset",
            "individual_tracks_are_sanity_checks_only": True,
        },
    }

# Backward-compatible alias, if later cells still use the older function name.
rank_songs_and_tanda_cot = rank_prompt_to_best_tanda_cot


## 8. Run test prompts

In [13]:
TEST_PROMPTS = [
    "energetic tango with strong rhythm for experienced dancers",
    "melancholic and slow, perfect for a late-night vals",
    "bright and cheerful milonga with clear melody",
    "dramatic tango with heavy bandoneon and dark mood",
    "smooth and relaxed, good for warming up the floor",
    "classic golden-age tango from the 40s, warm and nostalgic",
    "a lively vals from the 50s with a strong orchestra",
]

TOP_K = 10        # sanity check: top individual songs, keep between 5 and 10
BOTTOM_K = 5      # sanity check: bottom individual songs
TOP_TANDA_K = 5   # show top candidate tandas, primary result is rank 1

results = []

for prompt in TEST_PROMPTS:
    print("\n" + "=" * 100)
    print("PROMPT:", prompt)

    result = rank_prompt_to_best_tanda_cot(
        prompt=prompt,
        df=df_merged,
        top_k=TOP_K,
        bottom_k=BOTTOM_K,
        top_tanda_k=TOP_TANDA_K,
    )
    results.append(result)

    print("Hard constraints:", result["hard_constraints"])
    print("Selected 02a original numeric features:", list(result["feature_ranges_used"].keys()))

    print("\nPRIMARY OUTPUT — Best valid tanda by combo_key average score:")
    tanda = result["best_tanda"]
    if tanda is None:
        print("No valid tanda found under the current constraints.")
    else:
        print(
            f"combo_key={tanda['combo_key']} | style={tanda['style']} | "
            f"size={tanda['tanda_size']} | average_score={tanda['tanda_score']:.3f} | "
            f"available_in_combo={tanda['n_available_in_combo']}"
        )
        for j, tr in enumerate(tanda["tracks"], start=1):
            print(
                f"  {j}. {tr['score']:.3f} | {tr.get('title')} | "
                f"{tr.get('orchestra')} | {tr.get('singer')} | {tr.get('filename')}"
            )

    print("\nOther top candidate tandas:")
    for i, cand in enumerate(result["top_candidate_tandas"][1:], start=2):
        print(
            f"{i}. avg={cand['tanda_score']:.3f} | style={cand['style']} | "
            f"size={cand['tanda_size']} | combo={cand['combo_key']}"
        )

    print("\nSANITY CHECK ONLY — Top individual tracks regardless of combo:")
    for i, s in enumerate(result["sanity_check_top_k_songs"], start=1):
        print(
            f"{i:02d}. {s['score']:.3f} | {s.get('style')} | "
            f"{s.get('title')} | {s.get('orchestra')} | "
            f"{s.get('singer')} | combo={s.get('combo_key')}"
        )

    print("\nSANITY CHECK ONLY — Bottom individual tracks regardless of combo:")
    for i, s in enumerate(result["sanity_check_bottom_k_songs"], start=1):
        print(
            f"{i:02d}. {s['score']:.3f} | {s.get('style')} | "
            f"{s.get('title')} | {s.get('orchestra')} | "
            f"{s.get('singer')} | combo={s.get('combo_key')}"
        )

    print("Latency:", result["metadata"]["latency_seconds"])



PROMPT: energetic tango with strong rhythm for experienced dancers
Hard constraints: {'style': 'tango'}
Selected 02a original numeric features: ['bpm', 'onset_rate', 'danceability', 'average_loudness', 'dynamic_complexity']

PRIMARY OUTPUT — Best valid tanda by combo_key average score:
combo_key=enrique rodriguez | armando moreno | tango | style=tango | size=4 | average_score=1.000 | available_in_combo=6
  1. 1.000 | Como Has Cambiado Pebeta | Enrique Rodriguez | Armando Moreno | 07 como has cambiado pebeta.mp3
  2. 1.000 | El Encopao | Enrique Rodriguez | Armando Moreno | 09 el encopao.mp3
  3. 1.000 | En La Buena Y En La Mala | Enrique Rodriguez | Armando Moreno | 24 en la buena y en la mala.mp3
  4. 0.998 | Como Se Pianta La Vida | Enrique Rodriguez | Armando Moreno | 23 como se pianta la vida.mp3

Other top candidate tandas:
2. avg=0.999 | style=tango | size=4 | combo=pedro laurenz | juan carlos casas | tango
3. avg=0.998 | style=tango | size=4 | combo=miguel caló | raúl berón | t

## 9. Save results

In [14]:
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def jsonify(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, pd.Timestamp):
        return obj.isoformat()
    if isinstance(obj, dict):
        return {str(k): jsonify(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [jsonify(v) for v in obj]
    return obj

out_path = ARTIFACT_DIR / "cot_llm_02a_feature_map_combo_average_best_tanda_results.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(jsonify(results), f, indent=2, ensure_ascii=False)

print("Saved:", out_path)


Saved: drive/MyDrive/GenAI/proj/atdj_features_exp/cot_llm_02a_feature_map_combo_average_best_tanda_results.json
